# 08 · GR-source ablation — cross-model

The 09_end2end **Section 9** GR-source ablation ("what does the GR input buy?"),
lifted to a **head-to-head across three models** that all consume the same
`forward(dry, gr, params)` contract but use the GR curve differently:

| Model | run | GR mechanism | trained-on GR | **native** source |
|---|---|---|---|---|
| `gain_prior_ws` | `gain_prior_ws_20260703_114837` | multiplicative prior + waveshaper | oracle | `oracle` |
| `gr_tfilm` | `gr_tfilm_20260701_195700` | temporal FiLM (γ/β on hidden feats) | oracle | `oracle` |
| `e2e_cascade` | `e2e_predgr_20260705_185738` | multiplicative prior + WS, matched input | **predicted** | `predicted` |

Every model is streamed (deployment-exact, stateful, block-aligned) through the
**same** set of GR sources, then scored with the shared 9-column metric engine.
The two `amp_match_*` rows bypass the network entirely (`dry × 10^(gr/20)`) and
are model-independent — a shared floor.

> **Zero-init caveat.** `amp_match_pred` / `amp_match_oracle` are the zero-init
> operating point *only* for the multiplicative-prior models (`gain_prior_ws`,
> `e2e_cascade`). For `gr_tfilm` the GR is a FiLM conditioning signal, not a
> structural multiply, so the amp-match rows are just a reference floor, not
> that model's untrained baseline.

### Split caveat — why only 2 pairs
The e2e run uses the **v2 split** (val=`AncoraQui`, test = 5 external songs); the
other two use the older split (val=`Ecstasy`, test=`Air`,`AncoraQui`). To keep
the comparison honest, the cross-model table is scored **only on pairs held out
for all three models AND in no model's training set** — the clean intersection,
which is exactly the 2 low-threshold `AncoraQui` pairs (the same flavour as
Section 9's simplified 2-pair ablation). Per-model native-split numbers are shown
separately and are **not** cross-comparable.

In [1]:
# -- 0. Setup: three models + the frozen predicted-GR front-end --------
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from exp_common import (
    DEVICE, GR_DB_MAX, GR_DB_MIN, METRIC_COLS, SR,
    aggregate_rows, amplitude_match, ensure_eval_out, load_blackbox_gr,
    load_gain_prior, load_gain_prior_e2e, load_gr_tfilm, load_pair, load_split,
    pairs_from_keys, params_for, reference_gr_db, score_signal,
    stream_detector_gr, stream_gain_prior,
)

GP_WS_RUN   = "gain_prior_ws_20260703_114837_diffssl_lstm32_gain_prior_ws"
GR_TFILM_RUN = "gr_tfilm_20260701_195700_diffssl_lstm32_tvc_gr_tfilm"
E2E_RUN     = "e2e_predgr_20260705_185738_diffssl_lstm32_ws_predgr"

M_GPWS, HP_GPWS, DIR_GPWS = load_gain_prior(GP_WS_RUN)
M_GRTF, HP_GRTF, DIR_GRTF = load_gr_tfilm(GR_TFILM_RUN)
M_E2E,  HP_E2E,  DIR_E2E  = load_gain_prior_e2e(E2E_RUN)

# Predicted GR = the e2e cascade's own frozen stage-1 (pinned to its trained ckpt),
# so `predicted` here is byte-identical to the curve the e2e model was trained on.
FE = HP_E2E["gr_frontend"]
BB, _, _ = load_blackbox_gr(run_name=FE["run"], ckpt_name=FE["ckpt"])
print(f"predicted-GR front-end: {FE['run']} / {FE['ckpt']}")

# model registry: name -> (model, native GR source)
MODELS = {
    "gain_prior_ws": (M_GPWS, "oracle"),
    "gr_tfilm":      (M_GRTF, "oracle"),
    "e2e_cascade":   (M_E2E,  "predicted"),
}
NET_VARIANTS = ["oracle", "predicted", "const_mean", "const_0db"]
OUT = ensure_eval_out()

/Volumes/Saola's Drive/AllCode/thesis/Virtual-Analogue-Compressor-Modelling/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 8,418-param GainPriorWSDiffSSLLSTM  (gain_prior_ws_20260703_114837_diffssl_lstm32_gain_prior_ws / best-060-742980.ckpt)
Loaded 25,185-param GRTFiLMDiffSSLLSTM  (gr_tfilm_20260701_195700_diffssl_lstm32_tvc_gr_tfilm / best-039-487200.ckpt)
Loaded 8,418-param GainPriorWSE2ELSTM  (e2e_predgr_20260705_185738_diffssl_lstm32_ws_predgr / best-086-1305000.ckpt)  matched_input=True
Loaded 16,249-param blackbox_gr_lstm_film  (lstm_gr_20260705_125142_lstm_blackbox_gr_film / best-081-41000.ckpt)
predicted-GR front-end: lstm_gr_20260705_125142_lstm_blackbox_gr_film / best-081-41000.ckpt


In [2]:
# -- 0b. Clean shared pair set (held out for all 3, trained by none) ----
def _keys(run_dir):
    sm = load_split(run_dir)
    held = set(sm.val_pair_keys) | set(sm.test_pair_keys)
    return held, set(sm.train_pair_keys)

held, train = zip(*[_keys(d) for d in (DIR_GPWS, DIR_GRTF, DIR_E2E)])
CLEAN_KEYS = sorted(set.intersection(*held) - set.union(*train))
CLEAN_PAIRS = pairs_from_keys(CLEAN_KEYS)
print(f"clean shared pairs (held-out for all, trained by none): {len(CLEAN_PAIRS)}")
for s, st in CLEAN_PAIRS:
    print(f"  {s} | {st}")

clean shared pairs (held-out for all, trained by none): 2
  AncoraQui | threshold_-12_attack_10_release_0.4_ratio_10
  AncoraQui | threshold_-12_attack_1_release_0.1_ratio_2


## 1. GR sources (per pair)

Identical to Section 9's flipped ablation. `predicted` comes from the frozen
blackbox front-end (dry + knobs); `oracle` from `gr_curves/*.pt` (from wet);
`const_mean` is the whole-song oracle mean; `const_0db` disables the prior.

In [3]:
def gr_sources(setting, song, dry, wet, gr_oracle):
    T = dry.shape[-1]
    gr_pred = stream_detector_gr(BB, dry, params_for(setting), sample_len=T)
    n = min(T, gr_pred.shape[-1], gr_oracle.shape[-1])
    gr_pred, gr_oracle = gr_pred[..., :n], gr_oracle[..., :n]
    net = {
        "oracle":     gr_oracle,
        "predicted":  gr_pred,
        "const_mean": torch.full_like(gr_oracle, float(gr_oracle.mean())),
        "const_0db":  torch.zeros_like(gr_oracle),
    }
    bypass = {"amp_match_pred": gr_pred, "amp_match_oracle": gr_oracle}
    return net, bypass, n

## 2. Run — every (pair × model × network variant) + shared amp-match

Whole songs, stateful streaming (reset per pair, block-aligned chunks), same
protocol as the 05/06/09 eval notebooks. `amp_match_*` is streamed once per pair
(model-independent) and tagged `model="(bypass)"`.

In [ ]:
def run_clean():
    chunk_rows = {}   # (model, variant) -> [chunk rows]
    pair_rows = []
    for i, (song, setting) in enumerate(CLEAN_PAIRS, start=1):
        dry, wet, gr_oracle = load_pair(setting, song)
        net, bypass, n = gr_sources(setting, song, dry, wet, gr_oracle)
        dry, wet = dry[..., :n], wet[..., :n]
        p = params_for(setting)

        # model-independent amplitude-match floors
        for vname, gr_v in bypass.items():
            pred = amplitude_match(dry, gr_v)
            rows = score_signal(dry, pred, wet)
            chunk_rows.setdefault(("(bypass)", vname), []).extend(rows)
            pair_rows.append({"Model": "(bypass)", "Variant": vname, "Native": False,
                              "Song": song, "Setting": setting,
                              "Frames": sum(r["Frames"] for r in rows), **aggregate_rows(rows)})

        # every model through every network GR source
        for mname, (model, native) in MODELS.items():
            for vname in NET_VARIANTS:
                pred = stream_gain_prior(model, dry, net[vname], p)
                rows = score_signal(dry, pred, wet)
                chunk_rows.setdefault((mname, vname), []).extend(rows)
                pair_rows.append({"Model": mname, "Variant": vname, "Native": vname == native,
                                  "Song": song, "Setting": setting,
                                  "Frames": sum(r["Frames"] for r in rows), **aggregate_rows(rows)})
                print(f"[{i}/{len(CLEAN_PAIRS)}] {song[:12]:12s} {mname:13s} {vname:12s} "
                      f"GR MAE {pair_rows[-1]['GR MAE (dB)']:.3f}  MR-STFT {pair_rows[-1]['MR-STFT']:.3f}")
        del net, bypass
        gc.collect()

    agg = pd.DataFrame([
        {"Model": m, "Variant": v,
         "Duration (s)": sum(r["Frames"] for r in rows) / SR, **aggregate_rows(rows)}
        for (m, v), rows in chunk_rows.items()])
    return agg, pd.DataFrame(pair_rows)

clean_df, clean_pair_df = run_clean()
clean_df.round(4)

## 3. Headline — frame-weighted over the clean pairs

Rows are `model × variant`; the network's **native** operating point is where the
model was trained (`oracle` for the two oracle-GR models, `predicted` for the
cascade). Read down a column to compare GR sources within a model; read across
models at their native row for the deployable head-to-head.

In [ ]:
def pivot(df, metric):
    order = ["(bypass)", "gain_prior_ws", "gr_tfilm", "e2e_cascade"]
    vcols = ["amp_match_pred", "amp_match_oracle"] + NET_VARIANTS
    t = (df.pivot_table(index="Model", columns="Variant", values=metric)
           .reindex(index=order, columns=vcols))
    return t

for metric in ("GR MAE (dB)", "MR-STFT", "ESR (A-wt)", "M_NRMSE"):
    print(f"\n== {metric} ==")
    display(pivot(clean_df, metric).round(4))

## 4. Native operating point — deployable head-to-head

Each model at the GR source it was trained on, versus the two model-free
amplitude-match floors. This is the number that matters for deployment: the
cascade runs on `predicted`; the other two on `oracle` (an upper bound they
cannot reach at inference without a perfect GR estimate).

In [ ]:
native_rows = []
for mname, (_, native) in MODELS.items():
    r = clean_df[(clean_df.Model == mname) & (clean_df.Variant == native)].iloc[0]
    native_rows.append({"Model": mname, "GR source": native, **{c: r[c] for c in METRIC_COLS}})
for vname in ("amp_match_pred", "amp_match_oracle"):
    r = clean_df[(clean_df.Model == "(bypass)") & (clean_df.Variant == vname)].iloc[0]
    native_rows.append({"Model": vname, "GR source": vname.replace("amp_match_", ""),
                        **{c: r[c] for c in METRIC_COLS}})
native_df = pd.DataFrame(native_rows).set_index("Model")
display(native_df.round(4))

In [ ]:
# -- 4b. Cross-model bars: each variant, grouped by model ---------------
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
mods = list(MODELS)
colors = {"oracle": "#1f77b4", "predicted": "#d62728",
          "const_mean": "#2ca02c", "const_0db": "#7f7f7f"}
for ax, metric in zip(axes.ravel(), ("GR MAE (dB)", "MR-STFT", "ESR (A-wt)", "M_NRMSE")):
    xs = np.arange(len(mods)); w = 0.2
    for j, v in enumerate(NET_VARIANTS):
        vals = [pivot(clean_df, metric).loc[m, v] for m in mods]
        bars = ax.bar(xs + (j - 1.5) * w, vals, w, label=v, color=colors[v])
        # star the native operating point
        for k, m in enumerate(mods):
            if MODELS[m][1] == v:
                ax.plot(xs[k] + (j - 1.5) * w, vals[k], marker="*", ms=13,
                        color="gold", mec="k", mew=0.5, zorder=5)
    # amp-match floors as dashed h-lines
    for vname, ls in (("amp_match_oracle", "--"), ("amp_match_pred", ":")):
        fv = clean_df[(clean_df.Model == "(bypass)") & (clean_df.Variant == vname)][metric].iloc[0]
        ax.axhline(fv, color="k", lw=0.9, ls=ls, alpha=0.6, label=vname)
    ax.set_xticks(xs, mods, fontsize=9)
    ax.set_ylabel(metric); ax.grid(alpha=0.3, axis="y")
axes[0, 0].legend(fontsize=7, ncol=2, loc="upper left")
fig.suptitle("Cross-model GR-source ablation — ★ = native operating point, lines = amp-match floors")
fig.tight_layout()
fig.savefig(OUT / "08_crossmodel_gr_ablation_bars.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. GR-curve sweep — what each model reconstructs

One panel per **GR source** (`oracle`, `predicted`, `const_mean`, `const_0db`).
Each panel feeds that same source into **all three models** and overlays the GR
measured back from each model output against the target GR envelope (`gr_true`,
from dry/wet) and the GR input actually fed in. The first cell streams every
(source × model) pair once and caches the windowed curves; the plotting cell just
draws them, so re-styling is free (no re-streaming). Read a model's colour across
panels to see how its reconstruction degrades as the GR input drifts from oracle.

In [ ]:
def measure_gr(dry1d, sig1d):
    return np.clip(reference_gr_db(dry1d.double().numpy()[0], sig1d.double().numpy()[0]),
                   GR_DB_MIN, GR_DB_MAX)

SWEEP_SONG, SWEEP_SETTING = CLEAN_PAIRS[0]
START, DUR = 30.0, 6.0
dry, wet, gr_oracle = load_pair(SWEEP_SETTING, SWEEP_SONG)
net, bypass, n = gr_sources(SWEEP_SETTING, SWEEP_SONG, dry, wet, gr_oracle)
dry, wet = dry[..., :n], wet[..., :n]
a, b = int(START * SR), int((START + DUR) * SR)
t = np.arange(a, b) / SR
gr_true = measure_gr(dry, wet)[a:b]
p = params_for(SWEEP_SETTING)

# Stream every (GR source x model) once and cache the windowed GR curves so the
# plotting cell below can be re-styled without re-running the streaming.
sweep_input = {v: net[v][0, a:b].numpy() for v in NET_VARIANTS}
sweep_pred = {}
for v in NET_VARIANTS:
    for mname, (model, _) in MODELS.items():
        sweep_pred[(v, mname)] = measure_gr(dry, stream_gain_prior(model, dry, net[v], p))[a:b]
print(f"cached {len(sweep_pred)} curves ({len(NET_VARIANTS)} sources x {len(MODELS)} models)")

In [ ]:
# -- 5b. Sweep plot from cached curves — one panel per GR source -------
MODEL_COLORS = {"gain_prior_ws": "#d62728", "gr_tfilm": "#2ca02c", "e2e_cascade": "#9467bd"}
MODEL_LABELS = {
    "gain_prior_ws": "WS gain-prior (oracle-GR)",
    "gr_tfilm":      "gr_tfilm",
    "e2e_cascade":   "WS gain-prior (predicted-GR)",
}
fig, axes = plt.subplots(len(NET_VARIANTS), 1, figsize=(12, 2.0 * len(NET_VARIANTS)),
                         sharex=True, sharey=True, squeeze=False)
for ax, variant in zip(axes[:, 0], NET_VARIANTS):
    ax.plot(t, gr_true, lw=1.4, color="#1f77b4", label="target (dry/wet)")
    ax.plot(t, sweep_input[variant], lw=0.9, ls="--", color="#7f7f7f",
            alpha=0.8, label=f"GR input ({variant})")
    for mname in MODELS:
        ax.plot(t, sweep_pred[(variant, mname)], lw=1.1, color=MODEL_COLORS[mname],
                alpha=0.85, label=MODEL_LABELS[mname])
    ax.axhline(0, color="k", lw=0.5, alpha=0.4)
    ax.set_title(f"GR source = {variant}", fontsize=9, loc="left")
    ax.set_ylabel("GR (dB)"); ax.grid(alpha=0.3); ax.legend(fontsize=7, ncol=2, loc="lower right")
axes[-1, 0].set_xlabel("time (s)")
fig.suptitle(f"GR-source sweep — {SWEEP_SONG} | {SWEEP_SETTING} ({DUR:.0f}s @ {START:.0f}s)")
fig.tight_layout()
fig.savefig(OUT / "08_crossmodel_gr_sweep.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Per-model native split (context only — NOT cross-comparable)

Each model on **its own** full held-out split (val + test), at its native GR
source. Different pairs per model, so these are per-model sanity numbers, not a
head-to-head; the clean-intersection table above is the comparable one.

In [ ]:
def native_split_scores():
    out = []
    for mname, (model, native) in MODELS.items():
        rd = {"gain_prior_ws": DIR_GPWS, "gr_tfilm": DIR_GRTF, "e2e_cascade": DIR_E2E}[mname]
        sm = load_split(rd)
        pairs = pairs_from_keys(set(sm.val_pair_keys) | set(sm.test_pair_keys))
        rows = []
        for song, setting in pairs:
            dry, wet, gr_oracle = load_pair(setting, song)
            if native == "predicted":
                gr = stream_detector_gr(BB, dry, params_for(setting), sample_len=dry.shape[-1])
            else:
                gr = gr_oracle
            n = min(dry.shape[-1], gr.shape[-1], wet.shape[-1])
            pred = stream_gain_prior(model, dry[..., :n], gr[..., :n], params_for(setting))
            rows += score_signal(dry[..., :n], pred, wet[..., :n])
        out.append({"Model": mname, "native": native, "n_pairs": len(pairs),
                    "Duration (s)": sum(r["Frames"] for r in rows) / SR, **aggregate_rows(rows)})
        print(f"{mname:13s} native={native:9s} {len(pairs)} pairs done")
    return pd.DataFrame(out)

native_split_df = native_split_scores()
display(native_split_df.set_index("Model")[["native", "n_pairs"] + METRIC_COLS].round(4))

## 6b. Per-model VALIDATION vs TEST — split-resolved (native GR source)

The same split-scoring the 05/06/09 eval notebooks report, lifted across the
three models: each model on **its own** validation and test split, **separately**
(Section 6 above merges them), at its **native** GR source (`predicted` for the
cascade, `oracle` for the two oracle-GR models). Whole-song stateful streaming,
block-aligned chunks, scored with the shared 9-column engine and frame-weighted —
same protocol as `eval_lstm_gain_prior_ws.ipynb`'s `── 7` / `── 8b` cells, so the
numbers drop straight in.

> **Not cross-comparable across models** — each model has a different split (e2e
> uses the v2 split; its `test` is the 5 external `test_ground_truth` songs). Read
> down within a model (val vs test = generalisation gap); the clean-intersection
> table (Sections 3–4) is the head-to-head.

Memory-safe for the 16 GB Air: one pair is loaded, streamed, scored and freed
before the next; only the small per-chunk metric rows are retained. Emits a
split-level table plus a per-pair table, saved as CSVs.

In [ ]:
# -- 6b. Per-model validation & test metrics at the native GR source -----
# One pair at a time (load -> stream -> score -> free); models are already loaded.
MODEL_DIRS = {"gain_prior_ws": DIR_GPWS, "gr_tfilm": DIR_GRTF, "e2e_cascade": DIR_E2E}


def score_split(mname, model, native, pairs, split_name):
    """Whole-song stateful streaming over `pairs` -> (per-pair rows, split-agg row)."""
    pair_rows, chunk_rows = [], []
    for i, (song, setting) in enumerate(pairs, start=1):
        dry, wet, gr_oracle = load_pair(setting, song)
        p = params_for(setting)
        gr = (stream_detector_gr(BB, dry, p, sample_len=dry.shape[-1])
              if native == "predicted" else gr_oracle)
        n = min(dry.shape[-1], gr.shape[-1], wet.shape[-1])
        pred = stream_gain_prior(model, dry[..., :n], gr[..., :n], p)
        rows = score_signal(dry[..., :n], pred, wet[..., :n])
        chunk_rows += rows
        pair_rows.append({"Model": mname, "Split": split_name, "Song": song, "Setting": setting,
                          "Frames": sum(r["Frames"] for r in rows),
                          "Duration (s)": sum(r["Frames"] for r in rows) / SR,
                          **aggregate_rows(rows)})
        print(f"  [{split_name:10s} {i}/{len(pairs)}] {song[:14]:14s} {setting[:30]:30s} "
              f"GR MAE {pair_rows[-1]['GR MAE (dB)']:.3f}  MR-STFT {pair_rows[-1]['MR-STFT']:.3f}")
        del dry, wet, gr_oracle, gr, pred, rows
        gc.collect()
    split_row = {"Model": mname, "Split": split_name, "native": native, "Pairs": len(pairs),
                 "Duration (s)": sum(r["Frames"] for r in chunk_rows) / SR,
                 **aggregate_rows(chunk_rows)}
    return pair_rows, split_row


split_pair_rows, split_agg_rows = [], []
for mname, (model, native) in MODELS.items():
    sm = load_split(MODEL_DIRS[mname])
    val_pairs  = pairs_from_keys(sm.val_pair_keys)
    test_pairs = pairs_from_keys(sm.test_pair_keys)
    print(f"\n=== {mname}  (native={native})  —  val {len(val_pairs)}, test {len(test_pairs)} pairs ===")
    for split_name, pairs in (("validation", val_pairs), ("test", test_pairs)):
        if not pairs:
            print(f"  ({split_name}: no pairs — skipped)")
            continue
        pr, agg = score_split(mname, model, native, pairs, split_name)
        split_pair_rows += pr
        split_agg_rows.append(agg)
    gc.collect()

split_metrics_df = pd.DataFrame(split_agg_rows)
split_pair_df    = pd.DataFrame(split_pair_rows)
split_metrics_df.to_csv(OUT / "08_per_model_split_metrics.csv", index=False)
split_pair_df.to_csv(OUT / "08_per_model_split_metrics_pairs.csv", index=False)
print(f"\nSaved -> {OUT}/08_per_model_split_metrics(.csv, _pairs.csv)")
display(split_metrics_df.set_index(["Model", "Split"])[["native", "Pairs", "Duration (s)"] + METRIC_COLS].round(4))

In [ ]:
# -- 7. Save tables ----------------------------------------------------
clean_df.to_csv(OUT / "08_crossmodel_gr_ablation.csv", index=False)
clean_pair_df.to_csv(OUT / "08_crossmodel_gr_ablation_pairs.csv", index=False)
native_df.to_csv(OUT / "08_crossmodel_native_operating_point.csv")
native_split_df.to_csv(OUT / "08_crossmodel_native_split.csv", index=False)
print(f"Saved 4 CSVs + 2 PNGs -> {OUT}")